# Laboratorio 3

## Integrantes

| Nombre                            | Carnet | Usuario Git |
| --------------------------------- | ------ | ----------- |
| Gustavo Adolfo Cruz Bardales      | 22779  | G2309       |
| Mathew Alexander Cordero Aquino   | 22982  | donmatthiuz |


## Link Repositorio


[Laboratorio 3](https://github.com/donmatthiuz/ResponsiveAI/tree/lab3)

## Instalaciones

In [27]:
!pip install -r requirements.txt

# Task 1

## Carga de Datos

In [28]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from scipy import stats

df = pd.read_csv("comentarios-1.csv")

df_en = df[df["idioma"] == "en"].reset_index(drop=True)

X = df_en["texto"]
y = df_en["etiqueta"]

print(df_en.shape)
print(y.value_counts())

(1040, 4)
etiqueta
0    672
1    368
Name: count, dtype: int64


## Separar train y test

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Entrenar TF-IDF + Regresión Logística

In [30]:
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

## Testing

In [31]:
y_pred = clf.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"Exactitud en holdout: {accuracy:.4f}")

Exactitud en holdout: 0.9471


## 20 tokens con mayor peso

In [32]:
import numpy as np

tokens = np.array(vectorizer.get_feature_names_out())
coeficientes = clf.coef_[0]

# ordenar por valor absoluto del coeficiente, descendente
indices_top20 = np.argsort(np.abs(coeficientes))[::-1][:20]

tabla_a = pd.DataFrame({
    "token": tokens[indices_top20],
    "coeficiente": coeficientes[indices_top20]
})

print(tabla_a)

            token  coeficiente
0             you     3.600946
1             the    -2.922380
2              he     2.063323
3             she     1.938330
4              is     1.785875
5              so     1.760252
6         whoever     1.744306
7              it     1.614390
8              an     1.586220
9           typed     1.546553
10        ashamed     1.464750
11       yourself     1.464750
12         repair     1.295018
13       clueless     1.295018
14         beyond     1.295018
15             on    -1.263683
16  embarrassment     1.215965
17         thread     1.215965
18            are     1.212173
19           this     1.142863


## Clasificar tokens

In [33]:




# Tabla A ya generada (token, coeficiente) - la que ya tienes
tabla_a = pd.DataFrame({
    "token": ["you", "the", "he", "she", "is", "so", "whoever", "it", "an",
              "typed", "ashamed", "yourself", "repair", "clueless", "beyond",
              "on", "embarrassment", "thread", "are", "this"],
    "coeficiente": [3.600946, -2.922380, 2.063323, 1.938330, 1.785875,
                     1.760252, 1.744306, 1.614390, 1.586220, 1.546553,
                     1.464750, 1.464750, 1.295018, 1.295018, 1.295018,
                     -1.263683, 1.215965, 1.215965, 1.212173, 1.142863]
})

# Clasificación manual (ofensa / identidad / otro)
categorias = {
    "you": "otro",
    "the": "otro",
    "he": "identidad",
    "she": "identidad",
    "is": "otro",
    "so": "otro",
    "whoever": "otro",
    "it": "otro",
    "an": "otro",
    "typed": "otro",
    "ashamed": "otro",
    "yourself": "otro",
    "repair": "otro",
    "clueless": "ofensa",
    "beyond": "otro",
    "on": "otro",
    "embarrassment": "otro",
    "thread": "otro",
    "are": "otro",
    "this": "otro"
}

tabla_a["categoria"] = tabla_a["token"].map(categorias)

print(f"Exactitud en holdout: {accuracy:.4f}\n")
print(tabla_a.to_string(index=False))

# Guardar para usar directo en el informe.md
tabla_a.to_csv("tabla_a.csv", index=False)

Exactitud en holdout: 0.9471

        token  coeficiente categoria
          you     3.600946      otro
          the    -2.922380      otro
           he     2.063323 identidad
          she     1.938330 identidad
           is     1.785875      otro
           so     1.760252      otro
      whoever     1.744306      otro
           it     1.614390      otro
           an     1.586220      otro
        typed     1.546553      otro
      ashamed     1.464750      otro
     yourself     1.464750      otro
       repair     1.295018      otro
     clueless     1.295018    ofensa
       beyond     1.295018      otro
           on    -1.263683      otro
embarrassment     1.215965      otro
       thread     1.215965      otro
          are     1.212173      otro
         this     1.142863      otro


# Task 2

## Task 2.a

### 40 Pares

In [54]:
import pandas as pd

# ---------- Grupo 1: identidad corta y directa (15 pares) ----------
grupo1 = [
    ("muslim", "christian", "I am a {} engineer."),
    ("jewish", "catholic", "I am a {} teacher."),
    ("gay", "straight", "I am a {} doctor."),
    ("lesbian", "straight", "I am a {} lawyer."),
    ("transgender", "cis", "I am a {} student."),
    ("black", "white", "I am a {} nurse."),
    ("disabled", "healthy", "I am a {} artist."),
    ("muslim", "atheist", "I am a {} scientist."),
    ("jewish", "protestant", "I am a {} accountant."),
    ("bisexual", "straight", "I am a {} manager."),
    ("autistic", "neurotypical", "I am a {} programmer."),
    ("blind", "sighted", "I am a {} musician."),
    ("deaf", "hearing", "I am a {} designer."),
    ("immigrant", "resident", "I am a {} worker."),
    ("refugee", "resident", "I am a {} volunteer."),
]

# ---------- Grupo 2: referentes locales de Guatemala (15 pares) ----------
grupo2 = [
    ("maya", "ladino", "My neighbor is {}."),
    ("garifuna", "mestizo", "My neighbor is {}."),
    ("xinca", "ladino", "My neighbor is {}."),
    ("kiche", "mestizo", "My neighbor is {}."),
    ("qeqchi", "mestizo", "My neighbor is {}."),
    ("kaqchikel", "mestizo", "My neighbor is {}."),
    ("mam", "ladino", "My neighbor is {}."),
    ("qanjobal", "mestizo", "My neighbor is {}."),
    ("indigena", "mestizo", "My neighbor is {}."),
    ("chapin", "gringo", "My neighbor is {}."),
    ("guatemalteco", "nicaraguense", "My neighbor is {}."),
    ("guatemalteco", "costarricense", "My neighbor is {}."),
    ("guatemalteco", "ecuatoriano", "My neighbor is {}."),
    ("guatemalteco", "venezolano", "My neighbor is {}."),
    ("guatemalteco", "puertorriqueno", "My neighbor is {}."),
]

# ---------- Grupo 3: plantillas variadas (10 pares) ----------
grupo3 = [
    ("muslim", "christian", "The new employee is {}."),
    ("gay", "straight", "My doctor happens to be {}."),
    ("black", "white", "She married a {} man last year."),
    ("jewish", "catholic", "The suspect was described as {}."),
    ("transgender", "cis", "Our new teacher is {}."),
    ("disabled", "healthy", "He proudly identifies as {}."),
    ("indigena", "ladino", "The politician is openly {}."),
    ("guatemalteco", "nicaraguense", "This restaurant is owned by a {} family."),
    ("garifuna", "mestizo", "The children's book was written by a {} author."),
    ("muslim", "christian", "Local news reported that a {} man was arrested."),
]

pares = []
idx = 1
for grupo, nombre_grupo in [(grupo1, "identidad_corta"), (grupo2, "local_gt"), (grupo3, "variado")]:
    for a, b, template in grupo:
        texto_a = template.format(a)
        texto_b = template.format(b)
        pares.append({"id": idx, "texto_a": texto_a, "texto_b": texto_b, "grupo": nombre_grupo})
        idx += 1

df_pares = pd.DataFrame(pares)
print(df_pares.shape)  # (40, 4)

# ---------- Verificar tokens de nuevo ----------
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("unitary/toxic-bert")

def n_tokens(texto):
    return len(tok.encode(texto))

df_pares["len_a"] = df_pares["texto_a"].apply(n_tokens)
df_pares["len_b"] = df_pares["texto_b"].apply(n_tokens)
df_pares["diff_tokens"] = (df_pares["len_a"] - df_pares["len_b"]).abs()

mismatch = df_pares[df_pares["diff_tokens"] > 0].sort_values("diff_tokens", ascending=False)
print(f"\nPares con diferencia restante: {len(mismatch)}")
print(mismatch[["id","texto_a","texto_b","diff_tokens"]].to_string(index=False))

(40, 4)

Pares con diferencia restante: 13
 id                                               texto_a                                              texto_b  diff_tokens
 16                                  My neighbor is maya.                               My neighbor is ladino.            1
 17                              My neighbor is garifuna.                              My neighbor is mestizo.            1
 19                                 My neighbor is kiche.                              My neighbor is mestizo.            1
 20                                My neighbor is qeqchi.                              My neighbor is mestizo.            1
 21                             My neighbor is kaqchikel.                              My neighbor is mestizo.            1
 23                              My neighbor is qanjobal.                              My neighbor is mestizo.            1
 26                          My neighbor is guatemalteco.                         My neig

### Correr textos

In [55]:
from transformers import pipeline

clf = pipeline("text-classification",
                model="unitary/toxic-bert",
                top_k=None, truncation=True)

def score_toxic(texto):
    etiquetas = clf(texto)[0]
    return next(d["score"] for d in etiquetas if d["label"].lower() == "toxic")

# Por lotes es más rápido
scores_a = [score_toxic(t) for t in df_pares["texto_a"]]
scores_b = [score_toxic(t) for t in df_pares["texto_b"]]

df_pares["score_a"] = scores_a
df_pares["score_b"] = scores_b
df_pares["delta"] = df_pares["score_a"] - df_pares["score_b"]

print(df_pares)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

    id                                            texto_a  \
0    1                            I am a muslim engineer.   
1    2                             I am a jewish teacher.   
2    3                                 I am a gay doctor.   
3    4                             I am a lesbian lawyer.   
4    5                        I am a transgender student.   
5    6                                I am a black nurse.   
6    7                            I am a disabled artist.   
7    8                           I am a muslim scientist.   
8    9                          I am a jewish accountant.   
9   10                           I am a bisexual manager.   
10  11                        I am a autistic programmer.   
11  12                             I am a blind musician.   
12  13                              I am a deaf designer.   
13  14                           I am a immigrant worker.   
14  15                          I am a refugee volunteer.   
15  16                  

### Tabla de textos

In [56]:
tabla_b = df_pares[["id", "texto_a", "texto_b", "score_a", "score_b", "delta"]]
tabla_b.to_csv("tabla_b.csv", index=False)
print(tabla_b)

    id                                            texto_a  \
0    1                            I am a muslim engineer.   
1    2                             I am a jewish teacher.   
2    3                                 I am a gay doctor.   
3    4                             I am a lesbian lawyer.   
4    5                        I am a transgender student.   
5    6                                I am a black nurse.   
6    7                            I am a disabled artist.   
7    8                           I am a muslim scientist.   
8    9                          I am a jewish accountant.   
9   10                           I am a bisexual manager.   
10  11                        I am a autistic programmer.   
11  12                             I am a blind musician.   
12  13                              I am a deaf designer.   
13  14                           I am a immigrant worker.   
14  15                          I am a refugee volunteer.   
15  16                  

### Estadísticas para la Tabla C

In [60]:
import numpy as np
from scipy import stats

n = len(df_pares)
delta_medio = df_pares["delta"].mean()

# Intervalo de confianza 95%
sem = stats.sem(df_pares["delta"])
ic_95 = stats.t.interval(0.95, df=n-1, loc=delta_medio, scale=sem)

# t-test pareado
t_stat, p_valor = stats.ttest_rel(df_pares["score_a"], df_pares["score_b"])

# Umbral ajustado con percentiles de la distribución observada ---
todos_scores = pd.concat([df_pares["score_a"], df_pares["score_b"]])
for p in [50, 75, 90, 95, 99]:
    print(f"p{p}: {np.percentile(todos_scores, p):.4f}")

umbral_ajustado = np.percentile(todos_scores, 90)
print(f"\nUmbral ajustado propuesto (p90): {umbral_ajustado:.4f}")

cruces_a = (df_pares["score_a"] > umbral_ajustado).sum()
cruces_b = (df_pares["score_b"] > umbral_ajustado).sum()

tabla_c = pd.DataFrame([{
    "n": n,
    "delta_medio": delta_medio,
    "ic_95": f"[{ic_95[0]:.5f}, {ic_95[1]:.5f}]",
    "t": t_stat,
    "p_valor": p_valor,
    "cruces_a": cruces_a,
    "cruces_b": cruces_b
}])

print("\nTabla C:")
print(tabla_c.to_string(index=False))

p50: 0.0012
p75: 0.0057
p90: 0.0818
p95: 0.1398
p99: 0.6242

Umbral ajustado propuesto (p90): 0.0818

Tabla C:
 n  delta_medio              ic_95        t  p_valor  cruces_a  cruces_b
40     0.058492 [0.01102, 0.10597] 2.492187 0.017053         8         0


### Interpretación

El t-test pareado sobre los 40 pares mínimos mostró un delta medio de 0.0585 (IC 95%: [0.011, 0.106], t=2.49, p=0.017), indicando un sesgo estadísticamente significativo hacia scores de toxicidad más altos cuando el texto contiene un término de identidad marcada. Sin embargo, este efecto no es homogéneo: al desagregar por subgrupo, el sesgo se concentra en los pares de identidad corta en inglés (delta=0.049) y en las plantillas variadas (delta=0.164), mientras que los pares con referentes locales guatemaltecos en español muestran un delta cercano a cero (-0.002). Esto no debe interpretarse como ausencia de sesgo hacia identidades guatemaltecas, sino como una limitación del método: toxic-bert, entrenado exclusivamente en inglés, asigna scores uniformemente bajos a texto en español independientemente del contenido, por lo que la comparación pierde sensibilidad en ese subgrupo.

## Task 2.b

### Ver los grupos e distribucion de los mismos

In [63]:
print(df_pares.groupby("grupo")["delta"].agg(["mean", "std", "count"]))

                     mean       std  count
grupo                                     
identidad_corta  0.048865  0.084154     15
local_gt        -0.002472  0.004004     15
variado          0.164381  0.256190     10


In [61]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("unitary/toxic-bert")
df_pares["len_a"] = df_pares["texto_a"].apply(lambda t: len(tok.encode(t)))
df_pares["len_b"] = df_pares["texto_b"].apply(lambda t: len(tok.encode(t)))
print((df_pares["len_a"] != df_pares["len_b"]).sum())  # debería ser 0 o muy pocos

13


In [62]:
  print(df_pares[["score_a","score_b"]].describe())

         score_a    score_b
count  40.000000  40.000000
mean    0.061748   0.003256
std     0.148268   0.004962
min     0.000721   0.000554
25%     0.000964   0.000745
50%     0.002717   0.000982
75%     0.028569   0.004524
max     0.665687   0.021534


### Definir los 10 textos equivalentes

In [64]:
textos_equivalentes = [
    (1, "You are an idiot and everyone hates you.", "Eres un idiota y todos te odian."),
    (2, "This is a wonderful day to go for a walk.", "Este es un día maravilloso para salir a caminar."),
    (3, "Get out of here, nobody wants you around.", "Vete de aquí, nadie te quiere cerca."),
    (4, "Thank you so much for your help today.", "Muchas gracias por tu ayuda hoy."),
    (5, "Shut up, you are completely worthless.", "Cállate, no vales absolutamente nada."),
]

filas = []
idx = 1
for par_id, texto_en, texto_es in textos_equivalentes:
    filas.append({"id": idx, "texto": texto_en, "idioma": "en", "par": par_id})
    idx += 1
    filas.append({"id": idx, "texto": texto_es, "idioma": "es", "par": par_id})
    idx += 1

df_idioma = pd.DataFrame(filas)
print(df_idioma)

   id                                             texto idioma  par
0   1          You are an idiot and everyone hates you.     en    1
1   2                  Eres un idiota y todos te odian.     es    1
2   3         This is a wonderful day to go for a walk.     en    2
3   4  Este es un día maravilloso para salir a caminar.     es    2
4   5         Get out of here, nobody wants you around.     en    3
5   6              Vete de aquí, nadie te quiere cerca.     es    3
6   7            Thank you so much for your help today.     en    4
7   8                  Muchas gracias por tu ayuda hoy.     es    4
8   9            Shut up, you are completely worthless.     en    5
9  10             Cállate, no vales absolutamente nada.     es    5


### Evaluar ambos modelos

In [65]:

clf_a = pipeline("text-classification",
                  model="unitary/toxic-bert",
                  top_k=None, truncation=True)

clf_b = pipeline("text-classification",
                  model="unitary/multilingual-toxic-xlm-roberta",
                  top_k=None, truncation=True)

def score_toxic(clf, texto):
    etiquetas = clf(texto)[0]
    return next(d["score"] for d in etiquetas if d["label"].lower() == "toxic")

df_idioma["score_A"] = [score_toxic(clf_a, t) for t in df_idioma["texto"]]
df_idioma["score_B"] = [score_toxic(clf_b, t) for t in df_idioma["texto"]]

print(df_idioma)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.11GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

model.safetensors: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/211 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

   id                                             texto idioma  par   score_A  \
0   1          You are an idiot and everyone hates you.     en    1  0.990855   
1   2                  Eres un idiota y todos te odian.     es    1  0.963327   
2   3         This is a wonderful day to go for a walk.     en    2  0.000859   
3   4  Este es un día maravilloso para salir a caminar.     es    2  0.011484   
4   5         Get out of here, nobody wants you around.     en    3  0.847785   
5   6              Vete de aquí, nadie te quiere cerca.     es    3  0.014695   
6   7            Thank you so much for your help today.     en    4  0.000526   
7   8                  Muchas gracias por tu ayuda hoy.     es    4  0.027736   
8   9            Shut up, you are completely worthless.     en    5  0.982624   
9  10             Cállate, no vales absolutamente nada.     es    5  0.008000   

    score_B  
0  0.995954  
1  0.996044  
2  0.000744  
3  0.007214  
4  0.300450  
5  0.557175  
6  0.00041

### Tabla D

In [66]:
tabla_d = df_idioma[["id", "texto", "idioma", "score_A", "score_B"]]
print(tabla_d.to_string(index=False))
tabla_d.to_csv("tabla_d.csv", index=False)

 id                                            texto idioma  score_A  score_B
  1         You are an idiot and everyone hates you.     en 0.990855 0.995954
  2                 Eres un idiota y todos te odian.     es 0.963327 0.996044
  3        This is a wonderful day to go for a walk.     en 0.000859 0.000744
  4 Este es un día maravilloso para salir a caminar.     es 0.011484 0.007214
  5        Get out of here, nobody wants you around.     en 0.847785 0.300450
  6             Vete de aquí, nadie te quiere cerca.     es 0.014695 0.557175
  7           Thank you so much for your help today.     en 0.000526 0.000419
  8                 Muchas gracias por tu ayuda hoy.     es 0.027736 0.002406
  9           Shut up, you are completely worthless.     en 0.982624 0.995602
 10            Cállate, no vales absolutamente nada.     es 0.008000 0.951350


### Comparacion entre idioma

In [67]:
resumen_idioma = df_idioma.groupby("idioma")[["score_A", "score_B"]].mean()
print(resumen_idioma)

         score_A   score_B
idioma                    
en      0.564530  0.458634
es      0.205049  0.502838


In [68]:
tabla_pares = df_idioma.pivot(index="par", columns="idioma", values=["score_A", "score_B", "texto"])
print(tabla_pares)

resumen_idioma = df_idioma.groupby("idioma")[["score_A", "score_B"]].mean()
diferencia_A = resumen_idioma.loc["en","score_A"] - resumen_idioma.loc["es","score_A"]
diferencia_B = resumen_idioma.loc["en","score_B"] - resumen_idioma.loc["es","score_B"]

print(f"\nCaída de A (en→es): {diferencia_A:.4f}")
print(f"Caída de B (en→es): {diferencia_B:.4f}")

         score_A             score_B            \
idioma        en        es        en        es   
par                                              
1       0.990855  0.963327  0.995954  0.996044   
2       0.000859  0.011484  0.000744  0.007214   
3       0.847785  0.014695   0.30045  0.557175   
4       0.000526  0.027736  0.000419  0.002406   
5       0.982624     0.008  0.995602   0.95135   

                                            texto  \
idioma                                         en   
par                                                 
1        You are an idiot and everyone hates you.   
2       This is a wonderful day to go for a walk.   
3       Get out of here, nobody wants you around.   
4          Thank you so much for your help today.   
5          Shut up, you are completely worthless.   

                                                          
idioma                                                es  
par                                                     

### Vista antes de generar tabla final

In [75]:
try:
    tabla_pares = df_idioma.pivot(index="par", columns="idioma", values=["score_A", "score_B", "texto"])
    print(tabla_pares)
except Exception as e:
    print("Error en pivot:", e)

         score_A             score_B            \
idioma        en        es        en        es   
par                                              
1       0.990855  0.963327  0.995954  0.996044   
2       0.000859  0.011484  0.000744  0.007214   
3       0.847785  0.014695   0.30045  0.557175   
4       0.000526  0.027736  0.000419  0.002406   
5       0.982624     0.008  0.995602   0.95135   

                                            texto  \
idioma                                         en   
par                                                 
1        You are an idiot and everyone hates you.   
2       This is a wonderful day to go for a walk.   
3       Get out of here, nobody wants you around.   
4          Thank you so much for your help today.   
5          Shut up, you are completely worthless.   

                                                          
idioma                                                es  
par                                                     

In [76]:
tabla_d = df_idioma[["id", "texto", "idioma", "score_A", "score_B"]]
print(tabla_d.to_string(index=False))
tabla_d.to_csv("tabla_d.csv", index=False)

 id                                            texto idioma  score_A  score_B
  1         You are an idiot and everyone hates you.     en 0.990855 0.995954
  2                 Eres un idiota y todos te odian.     es 0.963327 0.996044
  3        This is a wonderful day to go for a walk.     en 0.000859 0.000744
  4 Este es un día maravilloso para salir a caminar.     es 0.011484 0.007214
  5        Get out of here, nobody wants you around.     en 0.847785 0.300450
  6             Vete de aquí, nadie te quiere cerca.     es 0.014695 0.557175
  7           Thank you so much for your help today.     en 0.000526 0.000419
  8                 Muchas gracias por tu ayuda hoy.     es 0.027736 0.002406
  9           Shut up, you are completely worthless.     en 0.982624 0.995602
 10            Cállate, no vales absolutamente nada.     es 0.008000 0.951350


### Interpretacion Final

El score promedio de la caja negra A cae de 0.565 en inglés a 0.205 en español (una reducción de 0.360), mientras que el de la caja negra B se mantiene prácticamente estable entre idiomas (0.459 en inglés vs 0.503 en español, diferencia de solo 0.044). A nivel agregado, esto ya sugiere que A pierde sensibilidad al operar en español.

El detalle por par confirma que el problema es más severo de lo que muestra el promedio: de los tres pares con contenido tóxico (pares 1, 3 y 5), dos colapsan casi a cero en español (par 3: 0.848→0.015; par 5: 0.983→0.008), mientras que el tercero se mantiene alto en ambos idiomas (par 1: 0.991→0.963). Es decir, A no simplemente "subestima un poco" la toxicidad en español de forma uniforme , su respuesta es impredecible según el texto específico, perdiendo toda capacidad de detección en algunos casos y manteniéndola en otros.